In [ ]:
# 파일을 읽기 위한 open 함수
from io import open

# 문자 정규화에 사용됨
# 예: é 같은 문자를 e로 바꾸는 데 사용
import unicodedata

# 문자열 관련 기능을 쓰기 위한 모듈
import string

# 정규표현식 모듈
# 문장에서 특수문자 제거, 공백 정리 등에 사용
import re

# 숫자 배열을 다루기 위한 라이브러리
# 여기서는 문장을 숫자 배열로 저장할 때 사용
import numpy as np

# 딥러닝 라이브러리 PyTorch
import torch

# PyTorch에서 데이터를 묶어서 학습용으로 불러오기 위한 도구들
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler


# GPU가 있으면 GPU를 사용하고, 없으면 CPU를 사용한다.
# cuda = NVIDIA GPU를 의미
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# 한 문장의 최대 단어 개수
# 여기서는 10개 미만의 짧은 문장만 사용한다.
MAX_LENGTH = 10


# 문장의 시작을 의미하는 특별한 숫자
# SOS = Start Of Sentence
SOS_token = 0

# 문장의 끝을 의미하는 특별한 숫자
# EOS = End Of Sentence
EOS_token = 1


# 영어 문장 중에서 아래 표현으로 시작하는 문장만 사용하기 위한 조건
# 예: "i am hungry", "he is tall", "you are nice" 같은 문장
eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)


# 언어 하나를 관리하는 클래스
# 예를 들어 프랑스어 단어장, 영어 단어장을 각각 만들 수 있다.
class Lang:
    def __init__(self, name):
        # 언어 이름 저장
        # 예: "eng", "fra"
        self.name = name

        # 단어를 숫자로 바꾸기 위한 딕셔너리
        # 예: {"je": 2, "suis": 3}
        self.word2index = {}

        # 단어가 몇 번 나왔는지 세는 딕셔너리
        # 예: {"je": 10, "suis": 7}
        self.word2count = {}

        # 숫자를 다시 단어로 바꾸기 위한 딕셔너리
        # 0은 문장 시작, 1은 문장 끝으로 미리 등록
        self.index2word = {
            SOS_token: "SOS",
            EOS_token: "EOS"
        }

        # 현재 등록된 단어 개수
        # SOS, EOS 두 개가 이미 있으므로 2부터 시작
        self.n_words = 2

    def addSentence(self, sentence):
        """
        문장 하나를 받아서 문장 안의 단어들을 단어장에 추가하는 함수
        예: "i am happy" → "i", "am", "happy" 각각 추가
        """

        # 문장을 공백 기준으로 단어 단위로 나눈다.
        for word in sentence.split(' '):
            # 단어 하나씩 단어장에 추가
            self.addWord(word)

    def addWord(self, word):
        """
        단어 하나를 단어장에 추가하는 함수
        처음 보는 단어면 새 번호를 붙이고,
        이미 있는 단어면 등장 횟수만 증가시킨다.
        """

        # 아직 단어장에 없는 단어라면
        if word not in self.word2index:
            # 단어에 번호를 붙인다.
            self.word2index[word] = self.n_words

            # 이 단어가 처음 등장했으므로 등장 횟수는 1
            self.word2count[word] = 1

            # 숫자로 단어를 찾을 수 있도록 반대 방향 딕셔너리에도 저장
            self.index2word[self.n_words] = word

            # 다음 단어는 다음 번호를 쓰도록 단어 개수를 1 증가
            self.n_words += 1

        # 이미 단어장에 있는 단어라면
        else:
            # 등장 횟수만 1 증가
            self.word2count[word] += 1


def filterPair(p):
    """
    문장 쌍 하나가 학습에 적합한지 검사하는 함수

    p[0] = 입력 문장
    p[1] = 정답 문장

    조건:
    1. 입력 문장이 MAX_LENGTH보다 짧아야 함
    2. 정답 문장도 MAX_LENGTH보다 짧아야 함
    3. 정답 문장이 eng_prefixes 중 하나로 시작해야 함
    """

    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)


def filterPairs(pairs):
    """
    전체 문장 쌍들 중에서 조건에 맞는 문장 쌍만 남기는 함수
    """

    return [pair for pair in pairs if filterPair(pair)]


def unicodeToAscii(s):
    """
    유니코드 문자를 일반 알파벳 형태로 바꾸는 함수

    예:
    "é" → "e"
    "ç" → "c"

    프랑스어처럼 악센트가 있는 문자를 단순한 영어 알파벳 형태로 바꾸기 위해 사용
    """

    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )


def normalizeString(s):
    """
    문장을 모델이 다루기 쉽게 정리하는 함수

    하는 일:
    1. 소문자로 바꿈
    2. 앞뒤 공백 제거
    3. 특수 악센트 제거
    4. 문장부호 앞뒤에 공백 추가
    5. 알파벳과 일부 문장부호 외에는 제거
    """

    # 소문자로 바꾸고, 앞뒤 공백 제거 후, 악센트 제거
    s = unicodeToAscii(s.lower().strip())

    # . ! ? 앞에 공백을 넣는다.
    # 예: "hello!" → "hello !"
    s = re.sub(r"([.!?])", r" \1", s)

    # 알파벳, '.', '!', '?'를 제외한 문자는 공백으로 바꾼다.
    # 예: 숫자, 특수기호 등 제거
    s = re.sub(r"[^a-zA-Z.!?]+", r" ", s)

    return s


def readLangs(lang1, lang2, reverse=False):
    """
    번역 데이터 파일을 읽고 문장 쌍을 만드는 함수

    예:
    lang1 = 'eng'
    lang2 = 'fra'

    data/eng-fra.txt 파일을 읽는다.

    reverse=True이면 문장 순서를 뒤집는다.
    즉, 영어 → 프랑스어 데이터가
    프랑스어 → 영어 데이터로 바뀐다.
    """

    print("Reading lines...")

    # data 폴더 안의 eng-fra.txt 같은 파일을 읽는다.
    # 파일 전체를 읽은 뒤 줄 단위로 나눈다.
    lines = open(
        'data/%s-%s.txt' % (lang1, lang2),
        encoding='utf-8'
    ).read().strip().split('\n')

    # 각 줄은 보통 "영어문장 \t 프랑스어문장" 형태이다.
    # 탭(\t)을 기준으로 나누고, 각각의 문장을 normalizeString으로 정리한다.
    pairs = [
        [normalizeString(s) for s in l.split('\t')]
        for l in lines
    ]

    # reverse=True이면 입력과 출력 문장을 서로 바꾼다.
    if reverse:
        # 예: [영어, 프랑스어] → [프랑스어, 영어]
        pairs = [list(reversed(p)) for p in pairs]

        # 입력 언어는 lang2가 됨
        input_lang = Lang(lang2)

        # 출력 언어는 lang1이 됨
        output_lang = Lang(lang1)

    # reverse=False이면 원래 순서 그대로 사용
    else:
        input_lang = Lang(lang1)
        output_lang = Lang(lang2)

    return input_lang, output_lang, pairs


def prepareData(lang1, lang2, reverse=False):
    """
    데이터를 학습에 사용할 수 있게 준비하는 전체 과정

    순서:
    1. 파일 읽기
    2. 문장 정리하기
    3. 너무 긴 문장 제거하기
    4. 필요한 영어 패턴만 남기기
    5. 단어장을 만들기
    """

    # 파일을 읽고 문장 쌍을 가져온다.
    input_lang, output_lang, pairs = readLangs(lang1, lang2, reverse)

    print("Read %s sentence pairs" % len(pairs))

    # 조건에 맞는 짧은 문장만 남긴다.
    pairs = filterPairs(pairs)

    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")

    # 남은 문장들을 보면서 입력 언어와 출력 언어의 단어장을 만든다.
    for pair in pairs:
        # 입력 문장의 단어들을 입력 언어 단어장에 추가
        input_lang.addSentence(pair[0])

        # 정답 문장의 단어들을 출력 언어 단어장에 추가
        output_lang.addSentence(pair[1])

    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)

    return input_lang, output_lang, pairs


def indexesFromSentence(lang, sentence):
    """
    문장을 숫자 리스트로 바꾸는 함수

    예:
    문장: "je suis"
    단어장: {"je": 2, "suis": 3}

    결과:
    [2, 3]
    """

    return [
        lang.word2index[word]
        for word in sentence.split(' ')
    ]


def tensorFromSentence(lang, sentence):
    """
    문장을 PyTorch 텐서로 바꾸는 함수

    텐서란?
    딥러닝 모델이 계산할 수 있는 숫자 데이터 형태라고 보면 된다.

    과정:
    1. 문장을 숫자 리스트로 바꿈
    2. 문장 끝에 EOS_token 추가
    3. PyTorch 텐서로 변환
    """

    # 문장을 단어 번호 리스트로 변환
    indexes = indexesFromSentence(lang, sentence)

    # 문장 끝 표시를 추가
    indexes.append(EOS_token)

    # 숫자 리스트를 PyTorch 텐서로 바꾼다.
    # dtype=torch.long은 정수형 숫자라는 뜻
    # device=device는 CPU 또는 GPU에 올린다는 뜻
    # view(-1, 1)은 세로 형태로 모양을 바꾸는 것
    return torch.tensor(
        indexes,
        dtype=torch.long,
        device=device
    ).view(-1, 1)


def tensorsFromPair(pair):
    """
    문장 쌍 하나를 텐서로 바꾸는 함수

    pair[0] = 입력 문장
    pair[1] = 정답 문장

    주의:
    이 함수는 input_lang, output_lang이라는 변수가
    함수 밖에 이미 만들어져 있어야 사용할 수 있다.
    """

    # 입력 문장을 텐서로 변환
    input_tensor = tensorFromSentence(input_lang, pair[0])

    # 정답 문장을 텐서로 변환
    target_tensor = tensorFromSentence(output_lang, pair[1])

    return input_tensor, target_tensor


def get_dataloader(batch_size):
    """
    학습용 DataLoader를 만드는 함수

    DataLoader란?
    데이터를 한꺼번에 전부 넣지 않고,
    batch_size만큼 조금씩 나누어 모델에게 전달해주는 도구이다.

    예:
    batch_size = 32이면
    문장 32개씩 묶어서 모델에 넣는다.
    """

    # eng-fra.txt 데이터를 읽는다.
    # reverse=True이므로 프랑스어 → 영어 번역 데이터가 된다.
    input_lang, output_lang, pairs = prepareData('eng', 'fra', True)

    # 전체 문장 쌍 개수
    n = len(pairs)

    # 입력 문장의 숫자 번호를 저장할 배열
    # 크기: 문장 개수 × 최대 문장 길이
    # 처음에는 전부 0으로 채운다.
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    # 입력 문장에서 실제 단어가 있는 위치를 표시할 배열
    # 실제 단어가 있으면 1, 빈칸이면 0
    input_mask = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    # 정답 문장의 숫자 번호를 저장할 배열
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    # 정답 문장에서 실제 단어가 있는 위치를 표시할 배열
    target_mask = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    # 문장 쌍을 하나씩 꺼내서 숫자 배열에 저장
    for idx, (inp, tgt) in enumerate(pairs):
        # 입력 문장을 숫자 리스트로 변환
        inp_ids = indexesFromSentence(input_lang, inp)

        # 정답 문장을 숫자 리스트로 변환
        tgt_ids = indexesFromSentence(output_lang, tgt)

        # 입력 문장의 숫자들을 input_ids 배열에 저장
        # 예: [5, 8, 12]라면 앞에서부터 3칸에 저장
        input_ids[idx, :len(inp_ids)] = inp_ids

        # 입력 문장에서 실제 단어가 있는 부분은 1로 표시
        input_mask[idx, :len(inp_ids)] = 1

        # 정답 문장의 숫자들을 target_ids 배열에 저장
        target_ids[idx, :len(tgt_ids)] = tgt_ids

        # 정답 문장에서 실제 단어가 있는 부분은 1로 표시
        target_mask[idx, :len(tgt_ids)] = 1

    # numpy 배열을 PyTorch 텐서로 바꾼다.
    # TensorDataset은 여러 개의 텐서를 하나의 데이터 묶음으로 만들어준다.
    train_data = TensorDataset(
        torch.LongTensor(input_ids).to(device),
        torch.LongTensor(input_mask).to(device),
        torch.LongTensor(target_ids).to(device),
        torch.LongTensor(target_mask).to(device)
    )

    # 데이터를 무작위 순서로 섞어서 가져오게 한다.
    # 학습할 때는 순서가 고정되어 있는 것보다 섞는 것이 일반적으로 좋다.
    train_sampler = RandomSampler(train_data)

    # DataLoader 생성
    # batch_size만큼 데이터를 묶어서 모델에 전달할 수 있게 해준다.
    train_dataloader = DataLoader(
        train_data,
        sampler=train_sampler,
        batch_size=batch_size
    )

    # 입력 언어 단어장, 출력 언어 단어장, 학습용 DataLoader 반환
    return input_lang, output_lang, train_dataloader

In [ ]:
# PyTorch 라이브러리 불러오기
# PyTorch는 딥러닝 모델을 만들고 학습시키는 데 사용하는 라이브러리이다.
import torch

# torch.nn은 신경망 모델을 만들 때 필요한 도구들을 모아둔 모듈이다.
# 예: RNN, GRU, Linear, Embedding 등
import torch.nn as nn

# optim은 모델 학습 시 가중치를 업데이트하는 최적화 도구이다.
# 예: Adam, SGD 등
# 이 코드 안에서는 아직 직접 사용되지는 않는다.
from torch import optim

# 자주 쓰는 함수들을 모아둔 모듈이다.
# 예: relu, softmax, log_softmax 등
import torch.nn.functional as F


# GPU가 있으면 GPU를 사용하고, 없으면 CPU를 사용한다.
# cuda는 NVIDIA GPU를 의미한다.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# =========================
# 1. Encoder
# =========================
# Encoder는 입력 문장을 읽는 역할을 한다.
# 예: 프랑스어 문장 "je suis etudiant"를 읽고
# 그 문장의 의미를 숫자 벡터 형태로 압축한다.
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        """
        input_size:
            입력 언어의 단어 개수
            예: 프랑스어 단어장이 5000개라면 input_size = 5000

        hidden_size:
            단어를 표현할 벡터 크기
            쉽게 말해, 단어 하나를 몇 개의 숫자로 표현할지 정하는 값
            예: hidden_size = 256이면 단어 하나를 숫자 256개로 표현
        """

        # 부모 클래스 nn.Module의 기능을 사용하기 위해 필요하다.
        super(EncoderRNN, self).__init__()

        # hidden_size를 클래스 내부에 저장
        self.hidden_size = hidden_size

        # Embedding은 단어 번호를 의미 있는 숫자 벡터로 바꿔준다.
        # 예:
        # "apple"이라는 단어가 35번이라면
        # 35라는 단순 숫자를 [0.12, -0.33, 0.51, ...] 같은 벡터로 바꾼다.
        self.embedding = nn.Embedding(input_size, hidden_size)

        # GRU는 문장을 순서대로 읽는 신경망이다.
        # RNN의 한 종류이며, 이전 단어의 정보를 기억하면서 다음 단어를 읽는다.
        #
        # 입력 크기: hidden_size
        # 출력 크기: hidden_size
        #
        # batch_first=True는 데이터 모양을
        # [문장 묶음 개수, 문장 길이, 단어 벡터 크기]
        # 형태로 사용하겠다는 의미이다.
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)

    def forward(self, input):
        """
        Encoder가 실제로 데이터를 처리하는 부분이다.

        input:
            숫자로 바뀐 입력 문장
            예: "je suis etudiant" → [12, 45, 103]
        """

        # 단어 번호를 단어 벡터로 바꾼다.
        # 예: [12, 45, 103] → 각각 의미를 가진 벡터로 변환
        embedded = self.embedding(input)

        # GRU가 문장을 순서대로 읽는다.
        #
        # output:
        #   문장의 각 단어를 읽고 난 결과
        #
        # hidden:
        #   문장 전체를 읽고 마지막에 남은 요약 정보
        output, hidden = self.gru(embedded)

        # Encoder의 결과를 반환한다.
        return output, hidden


# =========================
# 2. 일반 Decoder
# =========================
# Decoder는 Encoder가 읽은 정보를 바탕으로 번역 문장을 생성한다.
# 이 Decoder는 Attention이 없는 기본 Decoder이다.
class DecoderRNN(nn.Module):
    # Standard non-attentional decoder
    # Attention을 사용하지 않는 기본 Decoder라는 뜻
    def __init__(self, hidden_size, output_size):
        """
        hidden_size:
            Encoder와 Decoder 내부에서 사용할 벡터 크기

        output_size:
            출력 언어의 단어 개수
            예: 영어 단어장이 4000개라면 output_size = 4000
        """

        super(DecoderRNN, self).__init__()

        # 출력 단어 번호를 벡터로 바꾸기 위한 Embedding
        # Decoder는 이전에 생성한 단어를 보고 다음 단어를 예측한다.
        self.embedding = nn.Embedding(output_size, hidden_size)

        # Decoder도 GRU를 사용한다.
        # 이전 단어와 이전 상태를 참고해서 다음 단어를 예측한다.
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)

        # GRU의 결과를 실제 단어 개수만큼의 점수로 바꾼다.
        # 예: 영어 단어가 4000개라면, 각 단어가 정답일 점수 4000개를 만든다.
        self.out = nn.Linear(hidden_size, output_size)

    def forward(
        self,
        encoder_outputs,
        encoder_hidden,
        input_mask,
        target_tensor=None,
        SOS_token=0,
        max_len=10
    ):
        """
        Decoder가 문장을 생성하는 전체 과정이다.

        encoder_outputs:
            Encoder가 각 단어를 읽고 만든 결과들

        encoder_hidden:
            Encoder가 문장 전체를 읽고 만든 마지막 요약 정보

        input_mask:
            실제 단어 위치와 빈칸 위치를 구분하는 값
            이 기본 Decoder에서는 직접 사용하지 않는다.

        target_tensor:
            정답 문장
            학습할 때는 정답 문장을 알려주면서 학습할 수 있다.

        SOS_token:
            문장 시작 표시
            SOS = Start Of Sentence

        max_len:
            Decoder가 최대 몇 단어까지 만들지 정하는 값
        """

        # 현재 한 번에 처리하는 문장의 개수
        # 예: batch_size = 32이면 문장 32개를 동시에 처리
        batch_size = encoder_outputs.size(0)

        # Decoder의 첫 입력은 항상 SOS_token이다.
        # 즉, "문장을 시작해라"라는 신호를 넣어주는 것이다.
        decoder_input = torch.empty(
            batch_size,
            1,
            dtype=torch.long,
            device=device
        ).fill_(SOS_token)

        # Decoder의 처음 hidden 상태는 Encoder의 마지막 hidden 상태를 사용한다.
        # 쉽게 말해, Encoder가 읽고 요약한 내용을 Decoder에게 넘겨주는 것이다.
        decoder_hidden = encoder_hidden

        # Decoder가 각 시점마다 예측한 결과를 저장할 리스트
        decoder_outputs = []

        # 최대 max_len번 단어를 생성한다.
        for i in range(max_len):

            # 단어 하나를 예측한다.
            decoder_output, decoder_hidden = self.forward_step(
                decoder_input,
                decoder_hidden
            )

            # 예측 결과를 리스트에 저장한다.
            decoder_outputs.append(decoder_output)

            # target_tensor가 있다는 것은 학습 중이라는 뜻이다.
            if target_tensor is not None:
                # Teacher Forcing
                #
                # 모델이 방금 예측한 단어를 다음 입력으로 넣는 대신,
                # 정답 단어를 다음 입력으로 넣는다.
                #
                # 예:
                # 정답 문장이 "i am happy"라면
                # 모델이 틀리게 예측해도 다음 단계에는 정답인 "i"를 넣어준다.
                decoder_input = target_tensor[:, i].unsqueeze(1)

            else:
                # target_tensor가 없으면 실제 번역을 생성하는 상황이다.
                # 이때는 모델이 예측한 단어 중 점수가 가장 높은 단어를 선택한다.

                # topk(1)은 가장 점수가 높은 단어 1개를 고른다는 의미이다.
                topv, topi = decoder_output.data.topk(1)

                # 다음 입력으로 방금 예측한 단어를 넣는다.
                decoder_input = topi.squeeze(-1)

        # 리스트에 저장된 예측 결과들을 하나의 텐서로 합친다.
        # 결과 모양: [문장 개수, 생성한 단어 수, 출력 단어장 크기]
        decoder_outputs = torch.cat(decoder_outputs, dim=1)

        # 각 단어 후보에 대한 점수를 확률처럼 해석할 수 있게 바꾼다.
        # log_softmax는 학습에서 자주 쓰는 형태이다.
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)

        return decoder_outputs, decoder_hidden

    def forward_step(self, input, hidden):
        """
        Decoder가 단어 하나를 예측하는 과정이다.

        input:
            현재 Decoder에 들어가는 단어

        hidden:
            이전 단계까지의 기억 정보
        """

        # 현재 입력 단어 번호를 벡터로 바꾼다.
        output = self.embedding(input)

        # ReLU 활성화 함수 적용
        # 음수는 0으로 만들고, 양수는 그대로 둔다.
        # 신경망이 더 잘 학습하도록 도와준다.
        output = F.relu(output)

        # GRU에 현재 단어 벡터와 이전 hidden을 넣는다.
        output, hidden = self.gru(output, hidden)

        # GRU 결과를 단어장 크기만큼의 점수로 바꾼다.
        output = self.out(output)

        return output, hidden


# =========================
# 3. Bahdanau Attention
# =========================
# Attention은 Decoder가 번역할 때
# 입력 문장의 어떤 단어를 더 중요하게 볼지 정하는 장치이다.
#
# 예:
# 입력 문장: "je suis etudiant"
# 출력 단어를 만들 때
# "I"를 만들 때는 "je"에 집중하고,
# "student"를 만들 때는 "etudiant"에 집중하게 만든다.
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super(BahdanauAttention, self).__init__()

        # query를 변환하는 선형층
        # query는 Decoder의 현재 상태라고 보면 된다.
        self.W1 = nn.Linear(hidden_size, hidden_size)

        # values를 변환하는 선형층
        # values는 Encoder가 입력 문장의 각 단어를 읽고 만든 결과이다.
        self.W2 = nn.Linear(hidden_size, hidden_size)

        # Attention 점수를 하나의 숫자로 만들기 위한 선형층
        self.V = nn.Linear(hidden_size, 1)

        # 이 코드에서는 W3가 정의되어 있지만 실제로 사용되지는 않는다.
        self.W3 = nn.Linear(hidden_size, 1)

    def forward(self, query, values, mask):
        """
        Attention 계산 과정

        query:
            Decoder의 현재 상태
            지금 어떤 단어를 만들 차례인지에 대한 정보

        values:
            Encoder가 입력 문장의 각 단어를 읽고 만든 정보들

        mask:
            실제 단어와 빈칸을 구분하기 위한 값
            실제 단어 위치는 1, 빈칸은 0
        """

        # Additive Attention 방식
        #
        # Decoder의 현재 상태(query)와
        # Encoder의 각 단어 정보(values)를 비교해서
        # 어떤 입력 단어를 중요하게 볼지 점수를 계산한다.
        scores = self.V(
            torch.tanh(
                self.W1(query) + self.W2(values)
            )
        )

        # scores의 모양을 Attention 계산에 맞게 바꾼다.
        # [B, M, 1] → [B, 1, M]
        #
        # B = batch size, 즉 한 번에 처리하는 문장 개수
        # M = 입력 문장의 길이
        scores = scores.squeeze(2).unsqueeze(1)

        # 아래 코드는 다른 Attention 방식 예시이다.
        # 현재는 주석 처리되어 있어서 실행되지 않는다.

        # Dot-Product Attention:
        # Decoder 상태와 Encoder 결과를 내적해서 점수를 구하는 방식
        # scores = torch.bmm(query, values.permute(0,2,1))

        # Cosine Similarity:
        # 두 벡터의 방향이 얼마나 비슷한지로 점수를 구하는 방식
        # scores = F.cosine_similarity(query, values, dim=2).unsqueeze(1)

        # mask가 0인 위치, 즉 실제 단어가 아니라 빈칸인 위치는
        # Attention 대상에서 제외한다.
        #
        # -inf로 바꾸면 softmax를 거친 뒤 거의 0이 된다.
        # 즉, 빈칸에는 집중하지 않게 된다.
        scores.data.masked_fill_(
            mask.unsqueeze(1) == 0,
            -float('inf')
        )

        # softmax를 사용해 Attention 점수를 확률처럼 바꾼다.
        #
        # 예:
        # [2.0, 1.0, 0.1] 같은 점수를
        # [0.65, 0.25, 0.10]처럼 합이 1인 값으로 바꾼다.
        alphas = F.softmax(scores, dim=-1)

        # Attention 가중치를 사용해 Encoder 결과들을 가중합한다.
        #
        # 쉽게 말하면,
        # 중요한 단어의 정보는 많이 반영하고
        # 덜 중요한 단어의 정보는 적게 반영한다.
        context = torch.bmm(alphas, values)

        # context:
        #   현재 번역 단계에서 참고할 입력 문장의 핵심 정보
        #
        # alphas:
        #   입력 문장의 각 단어를 얼마나 중요하게 봤는지 나타내는 값
        return context, alphas


# =========================
# 4. Attention Decoder
# =========================
# Attention을 사용하는 Decoder이다.
# 기본 Decoder보다 더 똑똑하게 입력 문장의 중요한 부분을 참고하면서 번역한다.
class AttnDecoder(nn.Module):
    def __init__(self, hidden_size, output_size):
        """
        hidden_size:
            내부 벡터 크기

        output_size:
            출력 언어 단어장의 크기
        """

        super(AttnDecoder, self).__init__()

        # 출력 단어 번호를 벡터로 바꾸는 Embedding
        self.embedding = nn.Embedding(output_size, hidden_size)

        # Bahdanau Attention 사용
        self.attention = BahdanauAttention(hidden_size)

        # Attention Decoder의 GRU
        #
        # 입력 크기가 2 * hidden_size인 이유:
        # 1. 현재 Decoder 입력 단어의 embedding
        # 2. Attention으로 얻은 context
        #
        # 이 두 개를 붙여서 GRU에 넣기 때문이다.
        self.gru = nn.GRU(
            2 * hidden_size,
            hidden_size,
            batch_first=True
        )

        # GRU 결과를 출력 단어장 크기만큼의 점수로 변환
        self.out = nn.Linear(hidden_size, output_size)

    def forward(
        self,
        encoder_outputs,
        encoder_hidden,
        input_mask,
        target_tensor=None,
        SOS_token=0,
        max_len=10
    ):
        """
        Attention Decoder가 문장을 생성하는 전체 과정이다.
        """

        # 한 번에 처리하는 문장 개수
        batch_size = encoder_outputs.size(0)

        # Decoder의 첫 입력은 문장 시작 표시인 SOS_token
        decoder_input = torch.empty(
            batch_size,
            1,
            dtype=torch.long,
            device=device
        ).fill_(SOS_token)

        # Encoder의 마지막 hidden을 Decoder의 초기 hidden으로 사용
        decoder_hidden = encoder_hidden

        # Decoder의 각 단계 출력 결과를 저장할 리스트
        decoder_outputs = []

        # 최대 max_len개의 단어를 생성
        for i in range(max_len):

            # 단어 하나를 예측한다.
            # Attention Decoder는 단어 예측과 함께 Attention weight도 계산한다.
            decoder_output, decoder_hidden, attn_weights = self.forward_step(
                decoder_input,
                decoder_hidden,
                encoder_outputs,
                input_mask
            )

            # 현재 단계의 예측 결과 저장
            decoder_outputs.append(decoder_output)

            # 학습 중이면 정답 단어를 다음 입력으로 사용한다.
            if target_tensor is not None:
                # Teacher Forcing
                decoder_input = target_tensor[:, i].unsqueeze(1)

            else:
                # 실제 번역 생성 중이면 모델이 예측한 단어를 다음 입력으로 사용한다.
                topv, topi = decoder_output.data.topk(1)
                decoder_input = topi.squeeze(-1)

        # 각 단계의 출력을 하나로 합친다.
        decoder_outputs = torch.cat(decoder_outputs, dim=1)

        # 단어별 점수를 log 확률 형태로 바꾼다.
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)

        return decoder_outputs, decoder_hidden

    def forward_step(self, input, hidden, encoder_outputs, input_mask):
        """
        Attention Decoder가 단어 하나를 예측하는 과정이다.

        input:
            현재 Decoder에 들어가는 단어

        hidden:
            Decoder의 이전 기억 상태

        encoder_outputs:
            Encoder가 입력 문장의 각 단어를 읽고 만든 결과

        input_mask:
            실제 단어 위치와 빈칸 위치를 구분하는 값
        """

        # hidden의 모양을 Attention 계산에 맞게 바꾼다.
        #
        # GRU의 hidden 모양:
        # [층 개수, 문장 개수, hidden 크기]
        #
        # Attention에서 원하는 query 모양:
        # [문장 개수, 1, hidden 크기]
        query = hidden.permute(1, 0, 2)

        # Attention 계산
        #
        # context:
        #   지금 번역할 때 참고할 입력 문장의 핵심 정보
        #
        # attn_weights:
        #   입력 문장의 각 단어를 얼마나 중요하게 봤는지 나타내는 값
        context, attn_weights = self.attention(
            query,
            encoder_outputs,
            input_mask
        )

        # 현재 입력 단어를 벡터로 바꾼다.
        embedded = self.embedding(input)

        # 현재 단어 벡터와 Attention으로 얻은 context를 붙인다.
        #
        # embedded:
        #   Decoder가 현재 보고 있는 단어 정보
        #
        # context:
        #   Encoder 입력 문장에서 중요한 부분을 모은 정보
        #
        # 둘을 합쳐서 다음 단어 예측에 사용한다.
        attn = torch.cat((embedded, context), dim=2)

        # GRU가 현재 정보와 이전 hidden을 바탕으로 새 hidden을 만든다.
        output, hidden = self.gru(attn, hidden)

        # GRU 결과를 출력 단어장 크기만큼의 점수로 변환한다.
        output = self.out(output)

        return output, hidden, attn_weights


# =========================
# 5. Encoder + Decoder 전체 모델
# =========================
# EncoderDecoder는 Encoder와 Decoder를 하나로 묶은 최종 번역 모델이다.
#
# 전체 흐름:
# 1. Encoder가 입력 문장을 읽는다.
# 2. Decoder가 Encoder의 결과를 보고 번역 문장을 만든다.
class EncoderDecoder(nn.Module):
    def __init__(self, hidden_size, input_vocab_size, output_vocab_size):
        """
        hidden_size:
            단어와 문장을 표현하는 벡터 크기

        input_vocab_size:
            입력 언어 단어장의 크기

        output_vocab_size:
            출력 언어 단어장의 크기
        """

        super(EncoderDecoder, self).__init__()

        # 입력 문장을 읽는 Encoder 생성
        self.encoder = EncoderRNN(
            input_vocab_size,
            hidden_size
        )

        # Attention을 사용하는 Decoder 생성
        self.decoder = AttnDecoder(
            hidden_size,
            output_vocab_size
        )

        # 만약 Attention이 없는 기본 Decoder를 쓰고 싶다면
        # 위의 AttnDecoder 대신 아래 코드를 사용할 수 있다.
        # self.decoder = DecoderRNN(hidden_size, output_vocab_size)

    def forward(self, inputs, input_mask, targets=None):
        """
        전체 번역 모델의 실행 과정이다.

        inputs:
            숫자로 바뀐 입력 문장들

        input_mask:
            실제 단어와 빈칸을 구분하는 값

        targets:
            정답 문장
            학습 중이면 targets를 넣고,
            실제 번역 생성 시에는 targets 없이 사용한다.
        """

        # 1단계: Encoder가 입력 문장을 읽는다.
        encoder_outputs, encoder_hidden = self.encoder(inputs)

        # 2단계: Decoder가 Encoder의 결과를 보고 출력 문장을 만든다.
        decoder_outputs, decoder_hidden = self.decoder(
            encoder_outputs,
            encoder_hidden,
            input_mask,
            targets
        )

        # 최종 결과 반환
        return decoder_outputs, decoder_hidden

In [ ]:
# PyTorch 불러오기
# PyTorch는 딥러닝 모델을 만들고 학습시키는 라이브러리이다.
import torch

# torch.nn은 신경망을 만들 때 필요한 기능을 모아둔 모듈이다.
# 예: 손실함수, Linear, GRU, Embedding 등
import torch.nn as nn

# optim은 모델을 학습시킬 때 사용하는 최적화 알고리즘을 제공한다.
# 여기서는 Adam optimizer를 사용한다.
from torch import optim

# PyTorch에서 자주 쓰는 함수들을 모아둔 모듈이다.
# 예: softmax, relu 등
import torch.nn.functional as F

# 직접 만든 model.py 파일을 불러온다.
# model.py 안에는 Encoder, Decoder, Attention, EncoderDecoder 모델이 들어 있다.
import model

# 직접 만든 load_data.py 파일을 불러온다.
# load_data.py 안에는 데이터를 읽고 숫자로 바꾸고 DataLoader로 만드는 코드가 들어 있다.
import load_data


# GPU가 있으면 GPU를 사용하고, 없으면 CPU를 사용한다.
# cuda는 NVIDIA GPU를 의미한다.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# PAD_idx는 문장의 빈칸을 의미한다.
# 문장 길이를 맞추기 위해 비어 있는 자리에 0을 넣는다.
PAD_idx = 0

# SOS_token은 문장의 시작을 의미한다.
# SOS = Start Of Sentence
SOS_token = 0

# EOS_token은 문장의 끝을 의미한다.
# EOS = End Of Sentence
EOS_token = 1


# hidden_size는 모델 내부에서 단어와 문장을 표현하는 숫자 벡터의 크기이다.
# 쉽게 말하면, 단어 하나를 숫자 256개짜리 정보로 표현하겠다는 뜻이다.
hidden_size = 256

# batch_size는 한 번에 몇 개의 문장을 묶어서 학습할지 정하는 값이다.
# 여기서는 문장 32개씩 묶어서 모델에 넣는다.
batch_size = 32


def train(train_dataloader, model, n_epochs, learning_rate=0.0003):
    """
    모델을 여러 번 반복해서 학습시키는 함수이다.

    train_dataloader:
        학습 데이터를 batch_size만큼 나눠서 제공하는 도구

    model:
        학습시킬 번역 모델

    n_epochs:
        전체 학습 데이터를 몇 번 반복해서 학습할지 정하는 값
        예: n_epochs=20이면 전체 데이터를 20번 반복 학습

    learning_rate:
        모델이 한 번 학습할 때 얼마나 크게 수정될지 정하는 값
        값이 너무 크면 학습이 불안정하고,
        값이 너무 작으면 학습이 너무 느려진다.
    """

    # Adam optimizer 생성
    # optimizer는 모델의 가중치를 조금씩 수정하면서 성능을 높이는 역할을 한다.
    optimizer = optim.Adam(
        model.parameters(),
        lr=learning_rate
    )

    # 손실함수 생성
    # 손실함수는 모델의 예측이 정답과 얼마나 다른지 계산한다.
    #
    # NLLLoss는 log_softmax 결과와 함께 자주 사용되는 손실함수이다.
    #
    # ignore_index=PAD_idx:
    # PAD_idx에 해당하는 빈칸은 손실 계산에서 제외한다.
    # 즉, 문장 길이를 맞추기 위해 넣은 0은 정답 비교에서 무시한다.
    criterion = nn.NLLLoss(ignore_index=PAD_idx)

    # epoch는 전체 데이터를 몇 번째 반복 학습 중인지 의미한다.
    for epoch in range(1, n_epochs + 1):

        # 한 epoch 동안의 loss를 누적할 변수
        loss = 0

        # train_dataloader에서 batch를 하나씩 꺼낸다.
        # batch는 문장 여러 개를 묶어놓은 데이터이다.
        for iter, batch in enumerate(train_dataloader):

            # batch 안에는 다음 네 가지가 들어 있다.
            #
            # batch[0] = input_tensor
            #   입력 문장
            #
            # batch[1] = input_mask
            #   입력 문장에서 실제 단어가 있는 위치는 1,
            #   빈칸은 0으로 표시한 정보
            #
            # batch[2] = target_tensor
            #   정답 문장
            #
            # batch[3] = target_mask
            #   정답 문장의 실제 단어 위치 정보
            #   현재 코드에서는 사용하지 않는다.

            # 입력 문장
            # 모양: [B, SeqLen]
            # B = batch size, SeqLen = 문장 최대 길이
            input_tensor = batch[0]

            # 입력 문장의 mask
            # 실제 단어 위치와 빈칸 위치를 구분한다.
            input_mask = batch[1]

            # 정답 문장
            target_tensor = batch[2]

            # batch 하나를 학습시키고 나온 loss를 누적한다.
            loss += train_step(
                input_tensor,
                input_mask,
                target_tensor,
                model,
                optimizer,
                criterion
            )

        # 한 epoch가 끝날 때마다 평균 loss 출력
        # loss가 점점 줄어들수록 모델이 정답에 가까워지고 있다는 뜻이다.
        print('Epoch {} Loss {}'.format(epoch, loss / iter))


def train_step(input_tensor, input_mask, target_tensor, model, optimizer, criterion):
    """
    batch 하나를 학습시키는 함수이다.

    쉽게 말하면:
    1. 모델에게 입력 문장을 넣는다.
    2. 모델이 번역 문장을 예측한다.
    3. 예측과 정답을 비교해서 loss를 계산한다.
    4. loss를 바탕으로 모델을 조금 수정한다.
    """

    # 이전 학습에서 계산된 기울기값을 초기화한다.
    # PyTorch는 기본적으로 gradient를 계속 누적하기 때문에
    # 매번 학습 전에 0으로 비워줘야 한다.
    optimizer.zero_grad()

    # 모델에 입력 문장, 입력 mask, 정답 문장을 넣는다.
    #
    # target_tensor를 넣는 이유:
    # 학습할 때는 정답 문장을 참고하면서 학습하기 위해서이다.
    # 이것을 Teacher Forcing이라고 한다.
    decoder_outputs, decoder_hidden = model(
        input_tensor,
        input_mask,
        target_tensor
    )

    # decoder_outputs의 원래 모양:
    # [B, Seq, OutVoc]
    #
    # B:
    #   batch size, 한 번에 처리하는 문장 수
    #
    # Seq:
    #   출력 문장의 길이
    #
    # OutVoc:
    #   출력 언어 단어장의 크기
    #
    # 예:
    # [32, 10, 3000]
    # → 문장 32개, 각 문장 최대 10단어,
    #   각 위치마다 3000개 단어 중 어떤 단어인지 점수를 냄

    # 손실함수는 보통
    # [전체 단어 개수, 단어장 크기]
    # 형태를 원한다.
    #
    # 그래서 [B, Seq, OutVoc]를 [B * Seq, OutVoc]로 펼친다.
    #
    # target_tensor도 [B, Seq]에서 [B * Seq]로 펼친다.
    loss = criterion(
        decoder_outputs.view(
            -1,
            decoder_outputs.size(-1)
        ),
        target_tensor.view(-1)
    )

    # loss를 기준으로 역전파를 수행한다.
    # 역전파는 모델의 어떤 부분을 얼마나 고쳐야 하는지 계산하는 과정이다.
    loss.backward()

    # optimizer가 계산된 값을 바탕으로 모델의 가중치를 실제로 수정한다.
    optimizer.step()

    # loss.item()은 PyTorch tensor 형태의 loss를 일반 숫자로 바꾸는 것이다.
    return loss.item()


def ids2words(lang, ids):
    """
    숫자로 된 문장을 다시 단어로 바꾸는 함수이다.

    예:
    ids = [5, 8, 13]
    lang.index2word = {
        5: "je",
        8: "suis",
        13: "etudiant"
    }

    결과:
    ["je", "suis", "etudiant"]
    """

    return [
        lang.index2word[idx]
        for idx in ids
    ]


def greedy_decode(model, dataloader, input_lang, output_lang):
    """
    학습된 모델이 실제로 어떤 번역을 하는지 확인하는 함수이다.

    greedy_decode란?
        매 순간 가장 점수가 높은 단어 하나를 선택하는 방식이다.

    쉽게 말하면:
        모델이 다음 단어 후보 중 가장 자신 있어 하는 단어를 고르는 방식이다.
    """

    # torch.no_grad()는 gradient 계산을 하지 않겠다는 뜻이다.
    #
    # 학습할 때는 gradient가 필요하지만,
    # 결과만 확인할 때는 필요 없다.
    #
    # 그래서 no_grad를 쓰면 메모리도 덜 쓰고 속도도 빨라진다.
    with torch.no_grad():

        # dataloader에서 batch 하나만 꺼낸다.
        # 결과 확인용이므로 전체 데이터를 다 볼 필요 없이
        # 첫 번째 batch만 확인한다.
        batch = next(iter(dataloader))

        # 입력 문장
        input_tensor = batch[0]

        # 입력 mask
        input_mask = batch[1]

        # 정답 문장
        target_tensor = batch[2]

        # 모델에 입력 문장과 mask만 넣는다.
        #
        # 여기서는 target_tensor를 넣지 않는다.
        # 즉, 정답을 알려주지 않고 모델이 스스로 번역하게 한다.
        decoder_outputs, decoder_hidden = model(
            input_tensor,
            input_mask
        )

        # decoder_outputs에는 각 단어 위치마다
        # 출력 단어장 전체에 대한 점수가 들어 있다.
        #
        # topk(1)은 그중 가장 점수가 높은 단어 1개를 고르는 것이다.
        topv, topi = decoder_outputs.topk(1)

        # topi에는 예측된 단어 번호가 들어 있다.
        # squeeze()는 불필요한 차원을 제거해서 보기 쉽게 만든다.
        decoded_ids = topi.squeeze()

        # batch 안에 있는 문장들을 하나씩 출력한다.
        for idx in range(input_tensor.size(0)):

            # 입력 문장의 숫자들을 실제 단어로 변환한다.
            input_sent = ids2words(
                input_lang,
                input_tensor[idx].cpu().numpy()
            )

            # 모델이 예측한 출력 문장을 실제 단어로 변환한다.
            output_sent = ids2words(
                output_lang,
                decoded_ids[idx].cpu().numpy()
            )

            # 정답 문장을 실제 단어로 변환한다.
            target_sent = ids2words(
                output_lang,
                target_tensor[idx].cpu().numpy()
            )

            # 입력 문장 출력
            print('Input:  {}'.format(input_sent))

            # 정답 문장 출력
            print('Target: {}'.format(target_sent))

            # 모델이 예측한 문장 출력
            print('Output: {}'.format(output_sent))


# 이 부분은 이 파일을 직접 실행했을 때만 실행된다.
# 다른 파일에서 import할 때는 실행되지 않는다.
if __name__ == '__main__':

    # load_data.py의 get_dataloader 함수를 사용해서
    # 입력 언어 단어장, 출력 언어 단어장, 학습용 DataLoader를 가져온다.
    input_lang, output_lang, train_dataloader = load_data.get_dataloader(batch_size)

    # EncoderDecoder 번역 모델을 만든다.
    #
    # hidden_size:
    #   내부 벡터 크기
    #
    # input_lang.n_words:
    #   입력 언어 단어장의 단어 개수
    #
    # output_lang.n_words:
    #   출력 언어 단어장의 단어 개수
    #
    # .to(device):
    #   모델을 GPU 또는 CPU로 보낸다.
    model = model.EncoderDecoder(
        hidden_size,
        input_lang.n_words,
        output_lang.n_words
    ).to(device)

    # 모델을 20번 반복 학습한다.
    train(
        train_dataloader,
        model,
        n_epochs=20
    )

    # 학습이 끝난 뒤,
    # 모델이 실제로 어떤 번역을 하는지 확인한다.
    greedy_decode(
        model,
        train_dataloader,
        input_lang,
        output_lang
    )

ModuleNotFoundError: No module named 'model'